import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

df = pd.read_csv("/kaggle/input/datasets/mobeenfatimah/stroke-risk-prediction-dataset/stroke_risk_prediction_dataset.csv")

# Drop leakage columns + ID
X = df.drop(columns=["Stroke_Risk", "Stroke_Risk_Score", "AI_Health_Recommendation",
                     "Doctor_Consultation_Needed", "Patient_ID"])
y = df["Stroke_Risk"]

# Encode text columns, fill missing numerics
X = pd.get_dummies(X, drop_first=True).fillna(X.median(numeric_only=True))

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)
model.fit(X_tr, y_tr)
pred = model.predict(X_te)

print(f"Accuracy: {accuracy_score(y_te, pred):.4f} | Macro F1: {f1_score(y_te, pred, average='macro'):.4f}")
print(classification_report(y_te, pred))

# Top 10 features
print(pd.Series(model.feature_importances_, index=X.columns).nlargest(10))